## Question 1

What are the most common reasons for missing data in ETL pipelines?

**Answer:** Common reasons include source systems not capturing a value, optional fields being left blank, data-entry errors, failed API/database extraction, schema changes, file corruption, incomplete records, and data loss during integration or transformation.

## Question 2

Why is blindly deleting rows with missing values considered a bad practice in ETL?

**Answer:** Blind deletion can remove useful information, reduce the dataset, and introduce bias if the missing values are not random. The cause of missingness should be understood first, then a suitable method such as imputation, flagging, forward fill, or justified deletion should be chosen.

## Question 3

Explain the difference between Listwise deletion and Column deletion. Also mention one scenario where each is appropriate.

**Answer:** **Listwise deletion** removes entire rows that contain missing values in selected fields. It is appropriate when only a small number of rows are affected and deleting them will not meaningfully bias the dataset. **Column deletion** removes an entire column. It is appropriate when a column has a very high proportion of missing values and is not important for analysis.

## Question 4

Why is median imputation preferred over mean imputation for skewed data such as income?

**Answer:** Median imputation is preferred because the median is less affected by extreme values and outliers. Income data is often skewed by a few very high values, which can pull the mean upward, while the median remains more representative of a typical value.

## Question 5

What is forward fill and in what type of dataset is it most useful?

**Answer:** Forward fill replaces a missing value with the most recent previous non-missing value. It is most useful in ordered or time-based datasets where carrying the last observed value forward is reasonable.

## Question 6

Why should flagging missing values be done before imputation in an ETL workflow?

**Answer:** Flagging preserves information about which values were originally missing. After imputation, that information would otherwise be lost. A flag helps distinguish original values from imputed values and lets analysts study whether missingness itself has business meaning.

## Question 7

Consider a scenario where income is missing for many customers. How can this missingness itself provide business insights?

**Answer:** Missing income may reveal patterns such as certain customer groups being less willing to share income, specific regions having weak data collection, or some channels skipping the field. Studying missingness can therefore expose customer behavior or data-quality issues.

## Question 8

### Listwise Deletion

Remove all rows where `Region` is missing.

Tasks:
1. Identify affected rows
2. Show the dataset after deletion
3. Mention how many records were lost

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'Customer_ID': [101, 102, 103, 104, 105, 106, 107, 108],
    'Name': ['Rahul Mehta','Anjali Rao','Suresh Iyer','Neha Singh',
             'Amit Verma','Karan Shah','Pooja Das','Riya Kapoor'],
    'City': ['Mumbai','Bengaluru','Chennai','Delhi','Pune','Ahmedabad','Kolkata','Jaipur'],
    'Monthly_Sales': [12000, np.nan, 15000, np.nan, 18000, np.nan, 14000, 16000],
    'Income': [65000, np.nan, 72000, np.nan, 58000, 61000, np.nan, 69000],
    'Region': ['West','South','South','North',np.nan,'West','East','North']
})

affected_rows = df[df['Region'].isna()]
print("Affected rows:")
print(affected_rows.to_string(index=False))

df_after_deletion = df.dropna(subset=['Region'])
print("\nDataset after deletion:")
print(df_after_deletion.to_string(index=False))

records_lost = len(df) - len(df_after_deletion)
print("\nRecords lost:", records_lost)

**Answer:** The affected row is **Customer_ID 105 (Amit Verma)** because `Region` is missing. After deletion, **1 record is lost**.

## Question 9

### Imputation

Handle missing values in `Monthly_Sales` using Forward Fill.

Tasks:
1. Apply forward fill
2. Show before vs after values
3. Explain why forward fill is suitable here

In [ ]:
before_after = pd.DataFrame({
    'Customer_ID': df['Customer_ID'],
    'Before': df['Monthly_Sales']
})

df_ffill = df.copy()
df_ffill['Monthly_Sales'] = df_ffill['Monthly_Sales'].ffill()

before_after['After'] = df_ffill['Monthly_Sales']
print(before_after.to_string(index=False))

**Answer:** Forward fill replaces missing `Monthly_Sales` values with the previous available value.

- Customer 102 → **12000**
- Customer 104 → **15000**
- Customer 106 → **18000**

Forward fill is suitable when the rows are meaningfully ordered and the previous value is a reasonable estimate for the missing one.

## Question 10

### Flagging Missing Data

Create a flag column for missing `Income`.

Tasks:
1. Create `Income_Missing_Flag` (0 = present, 1 = missing)
2. Show updated dataset
3. Count how many customers have missing income

In [ ]:
df_flagged = df.copy()
df_flagged['Income_Missing_Flag'] = df_flagged['Income'].isna().astype(int)

print(df_flagged.to_string(index=False))

missing_income_count = int(df_flagged['Income_Missing_Flag'].sum())
print("\nCustomers with missing income:", missing_income_count)

**Answer:** Missing income is flagged as **1** for Customer_ID **102, 104, and 107**.

**Total customers with missing income: 3**